# Training a Network

On Monday we set a network's 9 weights by hand to classify XOR. Real networks have up to billions of weights or more. These networks  must *learn* their weight-values from data. In this lesson, we'll see how to apply **gradient descent** on a loss function, to learn the weights of a network like the one from Monday. In the following lab, you'll train a much larger, more complicated network to ingest images of hand-written digits using a similar process.

Logistic regression had one formula for its gradient (slopes along each beta-direction). A network's weights are organized in layers: we need a way to get the slope of the loss for every weight, at every layer. This process of using the relevant loss gradients to update a network's weights is called **backpropagation**.

### By the end, you'll be able to

1. Write the log-loss of a network with one sigmoid output, and explain why it's the same loss as logistic regression's (we'll cover multi-note output layer losses next week).
2. Estimate a slope of the loss by finite differences, and compute the cost of that approach for a whole network.
3. Derive the slopes of the loss for the output weights and for a hidden weight formally, using the chain rule, and explain why the method is called **backpropagation**.
4. Interpret a training run's loss curve, including a plateau and a run that gets stuck.
5. Explain what a minibatch and an epoch are, and why minibatch gradient descent is used.

> **How to read the code in this notebook.** There are two kinds of code cells.
>
> - **Code worth reading.** Cells that use standard libraries, such as numpy (`np.`), do everyday tasks you'll do yourself. Each is introduced with **Code worth reading** in bold. Read these line by line.
> - **Demo cells.** Cells that import from `ch_6_and_10_demos_helpers` draw a plot. That file was written to keep the notebook on ideas, so you never need to open it.

## Setup

We start by loading last lesson's XOR data: 100 points in four clusters, with class 1 at the $(0, 1)$ and $(1, 0)$ corners.

In [ ]:
import numpy as np

from ch_6_and_10_demos_helpers import xor_data

X_xor, y_xor = xor_data()        # Monday's XOR data: 100 points in four clusters
n = len(y_xor)
print('X_xor:', X_xor.shape, '  y_xor:', y_xor.shape)

In [ ]:
from ch_6_and_10_demos_helpers import plot_xor_training_data

plot_xor_training_data(X_xor, y_xor);

<a id="section-1"></a>

## 1 · Why the weights have to be learned

The first artificial neural networks were small enough to hand-tune (set really), the way we set Monday's XOR network. The perceptron (1958) added a rule for learning one step neuron's weights from its mistakes, but that rule doesn't extend to networks with hidden layers, and it's no longer used.

Modern networks are trained by **gradient descent**, as we discussed briefly on in the previous lesson ([Section 2's Ask](ch_6_and_10_02_neurons_to_networks.ipynb#section-2)): the loss has to change when a weight is nudged, so the activation has to have a slope (a defined derivative). The step is flat almost everywhere, so its slope is 0. Modern networks choose activation functions with well-behaved derivatives. Today, we'll use the sigmoid function.

<a id="section-2"></a>

## 2 · The network and its loss

In [ ]:
from ch_6_and_10_demos_helpers import plot_network_diagram

plot_network_diagram();

**The network.** It's Monday's network ([notation section](ch_6_and_10_02_neurons_to_networks.ipynb#notation)), with the sigmoid, written $\sigma$, as every activation. For one data point $(x_1, x_2)$, each line is one step of the forward pass:

$$
\begin{aligned}
z^{(1)}_k &= w^{(1)}_{k1}x_1 + w^{(1)}_{k2}x_2 + b^{(1)}_k && \text{hidden unit } k\text{'s score, for } k = 1, 2 \\
h_k &= \sigma\big(z^{(1)}_k\big) && \text{hidden unit } k\text{'s output} \\
z^{(2)} &= w^{(2)}_1 h_1 + w^{(2)}_2 h_2 + b^{(2)} && \text{the output unit's score} \\
p &= \sigma\big(z^{(2)}\big) && \text{the predicted probability of class 1}
\end{aligned}
$$

There are 9 parameters: 4 hidden weights, 2 hidden biases, 2 output weights, and 1 output bias.

**The loss.** The output $p$ is a probability, just like logistic regression's, so we score it the way Chapter 4 did. Chapter 4 chose the best curve by maximizing the **likelihood** ([Chapter 4, classification metrics demos, Section 9](../ch_4/ch_4_demos_2_classification_metrics.ipynb#section-9)): the product, over the data points, of the probability the model gave to what actually happened. Writing $p_i$ for $p(x_i)$:

$$
\ell = \prod_{i:\,y_i = 1} p_i \prod_{i:\,y_i = 0} \bigl(1 - p_i\bigr)
$$

To turn it into a loss, take its log (which turns the products into sums), flip the sign (so the best parameters sit at the loss's minimum instead of the likelihood's maximum), and divide by $n$ (an average, so the size doesn't depend on how many points there are):

$$
\begin{aligned}
L &= -\frac{1}{n}\ln \ell \\
&= -\frac{1}{n}\Big[\sum_{i:\,y_i = 1} \ln p_i + \sum_{i:\,y_i = 0} \ln(1 - p_i)\Big] \\
&= -\frac{1}{n}\sum_{i=1}^{n} \Big[\, y_i \ln p_i + (1 - y_i)\ln(1 - p_i) \Big]
\end{aligned}
$$

This is the **log-loss**. The last line looks different from the two sums above it, but it's the same. In the last line, each data point contributes the term

$$
y_i \ln p_i + (1 - y_i)\ln(1 - p_i)
$$

and $y_i$ is either 1 or 0. For a point with $y_i = 1$:

$$
\begin{aligned}
y_i \ln p_i + (1 - y_i)\ln(1 - p_i) &= 1 \cdot \ln p_i + 0 \cdot \ln(1 - p_i) \\
&= \ln p_i
\end{aligned}
$$

which is that point's term in the first sum. For a point with $y_i = 0$:

$$
\begin{aligned}
y_i \ln p_i + (1 - y_i)\ln(1 - p_i) &= 0 \cdot \ln p_i + 1 \cdot \ln(1 - p_i) \\
&= \ln(1 - p_i)
\end{aligned}
$$

which is that point's term in the second sum. So the labels $y_i$ and $1 - y_i$ act as switches: each point keeps exactly the term it had in the two-sum line. It's the loss you minimized with gradient descent in Chapter 4 ([gradient descent demos, Section 10](../ch_4/gradient_descent_demos.ipynb#section-10)), where the same steps are worked through one at a time.

Here $p_i$ is the network's predicted probability for data point $i$, and $y_i$ is its label, 1 or 0. Nothing about the loss is new. Only where $p_i$ comes from has changed: in Chapter 4 it was $\sigma(\beta_0 + \beta_1 x_i)$, and now it's the output of the forward pass above.

**Code worth reading.** The forward pass and the loss, for every data point at once, in Monday's pattern (`data @ W.T + b` for every layer). The 9 parameters live in a dictionary, one entry per weight matrix or bias vector, with the same shapes as Monday's.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def forward(params, X):
    "The hidden outputs H, shape (n, 2), and the output probabilities p, shape (n,)."
    H = sigmoid(X @ params['W1'].T + params['b1'])     # hidden layer
    Z2 = H @ params['W2'].T + params['b2']             # output layer's scores, shape (n, 1)
    p = sigmoid(Z2[:, 0])                              # the one output unit's probabilities
    return H, p


def loss(params, X, y):
    "The log-loss, averaged over the data points."
    _, p = forward(params, X)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


def random_params(seed):
    "Random starting weights and biases, drawn from a standard normal distribution."
    rng = np.random.default_rng(seed)
    return {'W1': rng.normal(size=(2, 2)),     # one row per hidden unit
            'b1': rng.normal(size=2),          
            'W2': rng.normal(size=(1, 2)),     # one row: one output unit
            'b2': rng.normal(size=1)}


params = random_params(seed=1)
print('loss at random starting weights:', round(loss(params, X_xor, y_xor), 3))

**What to notice:**

- **`y` and `p` are arrays, not single numbers.** In the math, $y_i$ and $p_i$ are one data point's label and probability. In the code, `y` holds all $n$ labels and `p` holds all $n$ probabilities, both with shape `(n,)`. Entry $i$ of `y` is $y_i$, and entry $i$ of `p` is $p_i$.
- **Elementwise operations/function on arrays** `np.log(p)` takes the log of every entry, so it holds $\ln p_1, \dots, \ln p_n$. `1 - y` subtracts every entry from 1. `y * np.log(p)` multiplies matching entries, giving $y_1 \ln p_1, \dots, y_n \ln p_n$. So the whole expression inside `np.mean(...)` is an array of $n$ numbers.
- **`np.mean` does the $\frac{1}{n}\sum$.** It adds up those $n$ terms and divides by $n$. With the minus sign in front, the line `-np.mean(...)` is exactly

  $$
  L = -\frac{1}{n}\sum_{i=1}^{n} \Big[\, y_i \ln p_i + (1 - y_i)\ln(1 - p_i) \Big]
  $$

  and `loss` returns a single number.
- **`sigmoid` works on whole arrays too,** for the same reason: `np.exp`, `+`, and `/` act on every entry. In `forward`, it's applied to an `(n, 2)` array of hidden scores, and it returns an `(n, 2)` array `H`: row $i$ is data point $i$'s $(h_1, h_2)$, and column $k$ is hidden unit $k$'s output for every point.
- **The bias vector:** `X @ params['W1'].T` has shape `(n, 2)`, and `params['b1']` has shape `(2,)`. NumPy adds `b1` to each of the $n$ rows (this is called **broadcasting**), so every data point gets each hidden unit's bias.
- **Why `Z2[:, 0]`?** The output layer's scores `Z2` have shape `(n, 1)`: one row per point, one column for the one output unit. `Z2[:, 0]` takes that column out as a plain `(n,)` array, the same shape as `y`. This matters: if `p` stayed `(n, 1)`, then `y * np.log(p)` would broadcast an `(n,)` array against an `(n, 1)` array into an `(n, n)` table, pairing every label with every probability. The loss would come out wrong, with no error message.
- **`_, p = forward(params, X)`.** `forward` returns two arrays, `H` and `p`. The loss only needs `p`, and the underscore is a convention for "unused value."
- **Why random starting weights?** If both hidden units started with identical weights, they would get identical slopes at every step and stay identical through every step of gradient descent. There are various strategies for initializing weights in various kinds of neural networks, but this simple strategy suffices for our purposes.

**Ask:** A network that knows nothing might output $p = 0.5$ for every data point. What would its loss be?


<details>
<summary>Answer</summary>

Start from the loss:

$$
L = -\frac{1}{n}\sum_{i=1}^{n} \Big[\, y_i \ln p_i + (1 - y_i)\ln(1 - p_i) \Big]
$$

Each data point contributes one term to the sum, the part in brackets:

$$
y_i \ln p_i + (1 - y_i)\ln(1 - p_i)
$$

Substitute $p_i = 0.5$. For a point with $y_i = 1$:

$$
\begin{aligned}
y_i \ln p_i + (1 - y_i)\ln(1 - p_i) &= 1 \cdot \ln 0.5 + 0 \cdot \ln(1 - 0.5) \\
&= \ln 0.5
\end{aligned}
$$

For a point with $y_i = 0$:

$$
\begin{aligned}
y_i \ln p_i + (1 - y_i)\ln(1 - p_i) &= 0 \cdot \ln 0.5 + 1 \cdot \ln(1 - 0.5) \\
&= \ln 0.5
\end{aligned}
$$

So every one of the $n$ terms is $\ln 0.5$, whatever the label:

$$
\begin{aligned}
L &= -\frac{1}{n}\sum_{i=1}^{n} \ln 0.5 \\
&= -\frac{1}{n} \cdot n \ln 0.5 \\
&= -\ln 0.5 \\
&= \ln 2 \\
&\approx 0.693
\end{aligned}
$$

The step from $-\ln 0.5$ to $\ln 2$ uses $-\ln a = \ln(1/a)$, and $1/0.5 = 2$.

Random starting weights land near this "knows nothing" value: the loss above is 0.701. Training should push the loss well below $\ln 2$.

</details>

<a id="section-3"></a>

## 3 · Review: gradient descent on a loss

This is a short reminder of Chapter 4's gradient descent demos ([Section 8](../ch_4/gradient_descent_demos.ipynb#demo-8a) and [Section 9](../ch_4/gradient_descent_demos.ipynb#section-9)).

- **The loss surface.** The loss turns each choice of parameters into one number, quantifying how well *that* model performs over the *entire* data set on which its being evaluated. 

- With two parameters, $(\beta_0, \beta_1)$, the loss forms a surface: one height above every point of the $(\beta_0, \beta_1)$ floor. For logistic regression's log-loss, the surface is a bowl. Training means finding (approximating really) the lowest point (minimum loss).
- **Slopes and the gradient.** Freeze every parameter but one, and the loss traces an ordinary curve. Its slope is the **partial derivative** with respect to that parameter, such as $\partial L / \partial \beta_1$. The **gradient** stacks every partial derivative into one vector:

$$
\nabla L(\beta) = \left( \frac{\partial L}{\partial \beta_0}, \; \frac{\partial L}{\partial \beta_1} \right)
$$

It points in the direction where the loss climbs fastest, so $-\nabla L$ is a vector that points the steepest way downhill.

The figure shows both views for a logistic regression on made-up data, with gradient descent's path from a poor starting point down to the bottom.

In [ ]:
from ch_6_and_10_demos_helpers import plot_gradient_descent_review

plot_gradient_descent_review();

**The update rule, three ways.** All three say the same thing.

In words, for each parameter ([Chapter 4, Section 6](../ch_4/gradient_descent_demos.ipynb#section-6)):

$$
\text{new value} = \text{old value} - \eta \times \text{slope of the loss with respect to that parameter}
$$

One parameter at a time, with $t$ counting the steps and $\beta^{(t)}$ the parameters after step $t$:

$$
\beta_j^{(t+1)} = \beta_j^{(t)} - \eta \, \frac{\partial L}{\partial \beta_j}\Big|_{\beta^{(t)}} \qquad \text{for every } j
$$

All at once, as vectors:

$$
\beta^{(t+1)} = \beta^{(t)} - \eta \, \nabla L\big(\beta^{(t)}\big)
$$

The vector form is the per-parameter form for every $j$ at once: entry $j$ of $\nabla L$ is $\partial L / \partial \beta_j$. The minus sign steps downhill, and the learning rate $\eta$ sets how far.

**For our network,** nothing changes except the number of parameters. Collect all 9 into one vector $\theta$ (ISL's letter for "all the parameters"). The update is

$$
\theta^{(t+1)} = \theta^{(t)} - \eta \, \nabla L\big(\theta^{(t)}\big) \qquad \text{The update rule } 
$$

and the gradient has 9 entries, one slope per parameter:

$$
\nabla L(\theta) = \left( \frac{\partial L}{\partial w^{(1)}_{11}}, \; \frac{\partial L}{\partial w^{(1)}_{12}}, \; \dots, \; \frac{\partial L}{\partial w^{(2)}_2}, \; \frac{\partial L}{\partial b^{(2)}} \right)
$$

We can't draw a 9-dimensional floor, but the picture above is still the right one: a surface over the parameters, and steps downhill.

**Two ways to get the gradient.** The update rule is the whole loop. What it needs at every step is $\nabla L$: all 9 slopes, at the current parameters. We'll compute them two ways:

- **Finite differences** (Section 4) measures each slope directly, by nudging one parameter at a time. It needs nothing but the loss, and it's easy to trust, but it's slow.
- **Backpropagation** (Section 5) uses calculus to get all 9 slopes at once from formulas. It's what every neural network library uses.

<a id="section-4"></a>

## 4 · The slopes, the direct way: finite differences

Before any calculus, there's a direct way to measure a slope, the same idea as Monday's Ask about nudging a weight. **Nudge one weight and see how much the loss moves.**

1. Hold every other parameter fixed.
2. Nudge the weight **up** by a tiny amount $h$, and compute the loss.
3. Nudge it **down** by $h$ from its original value, and compute the loss.
4. The slope is about the rise over the run:

$$
\frac{\partial L}{\partial w} \approx \frac{L(w + h) - L(w - h)}{2h}
$$

This is the definition of the derivative, with a small number $h$ in place of the limit. It's called **finite differences**. The picture below draws $h$ large so you can see the two nudged points. In code, $h$ is tiny, like 0.00001, and the line through the two points matches the true slope to many decimal places.

In [ ]:
from ch_6_and_10_demos_helpers import plot_finite_difference

plot_finite_difference(loss, params, X_xor, y_xor, name='W2', index=(0, 0), h=0.6, span=3.0);

**Code worth reading.** Finite differences for every parameter: one loop over every entry of every array. Each entry is nudged in place (the function **mutates** `params`), the two losses are computed, and the original value is put back.

In [ ]:
def numerical_gradient(params, X, y, h=1e-5):
    "The slope of the loss for every parameter, by finite differences."
    grads = {}
    for name, value in params.items():
        grads[name] = np.zeros_like(value)
        for index in np.ndindex(value.shape):          # every entry of this array
            original = value[index]
            value[index] = original + h
            loss_up = loss(params, X, y)
            value[index] = original - h
            loss_down = loss(params, X, y)
            value[index] = original                     # put it back
            grads[name][index] = (loss_up - loss_down) / (2 * h)
    return grads


numerical = numerical_gradient(params, X_xor, y_xor)
for name, g in numerical.items():
    print(name, g.round(4))

**Ask:** Each parameter needs two forward passes (one for each nudge). How many forward passes does one gradient take for our network? For a network with a million parameters? GPT-3 has 175 billion.


<details>
<summary>Answer</summary>

- **Our network:** 9 parameters, so 18 forward passes. That's nothing.
- **A million parameters:** 2 million forward passes, for **one** gradient descent step. Training takes many thousands of steps.
- **GPT-3:** 350 billion forward passes per step, each through a 175-billion-parameter network. Impossible.

Finite differences is correct but far too slow for real networks. We need the slopes from a formula, the way Chapter 4 had a formula for logistic regression's slopes. The next section derives one.

**Finite differences still has a job.** It's how you *check* a slope formula: if your formula and finite differences disagree, the formula (or its code) has a bug. PyTorch ships a function for exactly this, `torch.autograd.gradcheck`. In this notebook, finite differences is our answer key.

</details>

<a id="section-5"></a>

## 5 · The slopes from calculus: backpropagation

**Goal:** Gradient descent's update rule ([Section 3](#section-3)) needs the gradient of the loss at the current parameters. The gradient is the vector of the loss's partial derivatives with respect to all 9 parameters:

$$
\nabla L(\theta) = \left[\begin{array}{c} \begin{array}{c} \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial w^{(1)}_{11}} \\ \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial w^{(1)}_{12}} \\ \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial w^{(1)}_{21}} \\ \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial w^{(1)}_{22}} \end{array} \\
\begin{array}{c} \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial b^{(1)}_1} \\ \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial b^{(1)}_2} \end{array} \\
\begin{array}{c} \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial w^{(2)}_1} \\ \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial w^{(2)}_2} \end{array} \\
\begin{array}{c} \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial b^{(2)}} \end{array} \end{array}\right]
\begin{array}{l} \left.\vphantom{\begin{array}{c} \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial w^{(1)}_{11}} \\ \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial w^{(1)}_{12}} \\ \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial w^{(1)}_{21}} \\ \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial w^{(1)}_{22}} \end{array}}\right\}\ \text{hidden-layer weights, } W^{(1)} \\
\left.\vphantom{\begin{array}{c} \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial b^{(1)}_1} \\ \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial b^{(1)}_2} \end{array}}\right\}\ \text{hidden-layer biases, } b^{(1)} \\
\left.\vphantom{\begin{array}{c} \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial w^{(2)}_1} \\ \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial w^{(2)}_2} \end{array}}\right\}\ \text{output weights, } W^{(2)} \\
\left.\vphantom{\begin{array}{c} \rule[-12pt]{0pt}{32pt}\dfrac{\partial L}{\partial b^{(2)}} \end{array}}\right\}\ \text{output bias, } b^{(2)} \end{array}
$$

This section derives a formula for each entry, in this order:

$$
\begin{aligned}
&\frac{\partial L}{\partial w^{(2)}_1},\; \frac{\partial L}{\partial w^{(2)}_2},\; \frac{\partial L}{\partial b^{(2)}} && \text{Step 1: the output layer} \\
&\frac{\partial L}{\partial w^{(1)}_{21}} && \text{Step 2: one hidden weight, by the chain rule} \\
&\frac{\partial L}{\partial w^{(1)}_{11}},\; \frac{\partial L}{\partial w^{(1)}_{12}},\; \frac{\partial L}{\partial w^{(1)}_{22}},\; \frac{\partial L}{\partial b^{(1)}_1},\; \frac{\partial L}{\partial b^{(1)}_2} && \text{Step 3: the rest, by substitution}
\end{aligned}
$$

Step 4 then plugs in our loss, and the code computes all 9 at once.

**Setup: one data point at a time.** The loss is always an average over the $n$ data points of one loss per point, $L_i$ for point $i$ (Section 2's log-loss has this form). The slope of an average is the average of the slopes, so for any parameter $\theta_m$:

$$
\begin{aligned}
L &= \frac{1}{n}\sum_{i=1}^{n} L_i \\
\frac{\partial L}{\partial \theta_m} &= \frac{1}{n}\sum_{i=1}^{n} \frac{\partial L_i}{\partial \theta_m}
\end{aligned}
$$

So we derive the slopes for **one data point's** loss and average them at the end. To keep the notation light, from here on $L$ means that one point's loss, and $p$, $y$, $h_k$, $x_j$ are that point's values.

**We leave the loss general.** For one data point, the loss is some function of the prediction, $L(p)$ (the label $y$ is fixed for that point). Section 2's log-loss is one choice. Steps 1 to 3 work for any loss, and Step 4 plugs in log-loss.

### Step 1: the output layer

**This step's entries of the gradient** ([listed at the top of this section](#section-5)):

$$
\dfrac{\partial L}{\partial w^{(2)}_1},\qquad \dfrac{\partial L}{\partial w^{(2)}_2},\qquad \dfrac{\partial L}{\partial b^{(2)}}
$$

We derive the two weights' entries together, writing $w^{(2)}_k$ for $k = 1, 2$, and then the bias's.

The output unit's score and output are

$$
\begin{aligned}
z^{(2)} &= w^{(2)}_1 h_1 + w^{(2)}_2 h_2 + b^{(2)} \\
p &= \sigma\big(z^{(2)}\big) \\
L(p) &= \text{Some loss function that depends on the network output, p}
\end{aligned}
$$

Notice: Three kinds of operation
- a weighted sum, 
- a sigmoid, and 
- the loss. 

Every network is built from these pieces, and each kind has one rule for its derivative.

$L$ is a function of $p$, which is a function of $z^{(2)}$, which is a function of $w^{(2)}_k$. Thinking of $z^{(2)}$ as a function of $w^{(2)}_k$ might seem strange, but gradient descent needs derivatives with respect to the weights, so here the weight is the variable. Written as functions inside functions:

$$
L\Big(p\big(z^{(2)}(w^{(2)}_k)\big)\Big)
$$

The chain rule peels off one function at a time, from the outside in:

$$
\frac{\partial L}{\partial w^{(2)}_k} = \frac{\partial L}{\partial p} \cdot \frac{\partial p}{\partial z^{(2)}} \cdot \frac{\partial z^{(2)}}{\partial w^{(2)}_k} \qquad \text{(1)}
$$

The three **local derivatives** (link slopes), one for each function in the chain:

$$
\begin{aligned}
\frac{\partial L}{\partial p} & && \text{left general for now; with the next line, it forms the error signal } \delta^{(2)} \\
\frac{\partial p}{\partial z^{(2)}} &= p\,(1 - p) && \text{a sigmoid: output} \times (1 - \text{output}) \text{ (optional derivation below)} \\
\frac{\partial z^{(2)}}{\partial w^{(2)}_k} &= h_k && \text{a weighted sum, by a weight: the input that weight multiplies}
\end{aligned}
$$

Plugging these into (1):

$$
\frac{\partial L}{\partial w^{(2)}_k} = \underbrace{\frac{\partial L}{\partial p}\, p\,(1 - p)}_{\delta^{(2)}}\; h_k
$$

The first two factors are the slope of the loss with respect to the output score, $\partial L / \partial z^{(2)}$. They're the same for every output parameter, so give them a name, the output unit's **error signal**:

$$
\delta^{(2)} = \frac{\partial L}{\partial z^{(2)}} = \frac{\partial L}{\partial p}\, p\,(1 - p)
$$

$\delta^{(2)}$ is the only place the choice of loss enters. With it,

$$
\frac{\partial L}{\partial w^{(2)}_k} = \underbrace{\delta^{(2)}}_{\substack{\phantom{x} \\ \llap{\text{error signal}}}} \cdot \underbrace{h_k}_{\substack{\phantom{x} \\ \rlap{\text{input to layer 2}}}}
$$

Here "input" means the value the weight multiplies: $h_k$ is an input to the output layer (layer 2), not an input to the network. This shape, (error signal) × (the input to the weight's layer), comes back for every weight in Step 3, where the hidden layer's inputs are the network's inputs $x_j$.

For the bias, only the last local derivative changes:

$$
\frac{\partial z^{(2)}}{\partial b^{(2)}} = 1
$$

Plugging this into $\dfrac{\partial L}{\partial b^{(2)}} = \dfrac{\partial L}{\partial p} \cdot \dfrac{\partial p}{\partial z^{(2)}} \cdot \dfrac{\partial z^{(2)}}{\partial b^{(2)}}$, we get (equivalent to replacing $h_k$ with 1 above)

$$
\frac{\partial L}{\partial b^{(2)}} = \delta^{(2)}
$$

(Each local derivative follows the rule for one kind of operation, and each uses only numbers the forward pass has already produced: $p$ and $h_k$. The same rules come back in Step 2.)


<details>
<summary>Optional: deriving the sigmoid's derivative</summary>

For $p = \sigma(z) = \dfrac{1}{1 + e^{-z}}$:

$$
\begin{aligned}
\frac{dp}{dz} &= \frac{d}{dz}\big(1 + e^{-z}\big)^{-1} \\
&= -\big(1 + e^{-z}\big)^{-2} \cdot \big(-e^{-z}\big) \\
&= \frac{e^{-z}}{\big(1 + e^{-z}\big)^2} \\
&= \frac{1}{1 + e^{-z}} \cdot \frac{e^{-z}}{1 + e^{-z}} \\
&= p\,(1 - p)
\end{aligned}
$$

The last line uses $\dfrac{e^{-z}}{1 + e^{-z}} = 1 - \dfrac{1}{1 + e^{-z}} = 1 - p$.

</details>

### One formula isn't the goal

We could keep going this way: derive each hidden weight's slope by hand, then code each formula. But those formulas would belong to **this one network**. Add a hidden unit or a layer, swap the sigmoid for another activation, or change the loss, and every formula changes. Real networks have millions of parameters, complicated structures and their designs change frequently. 

**Nobody derives their slopes by hand.**

What we need is an **algorithm** that:

- works for **any** network built from pieces we can calculate, regardless of network topology: scores, activations, and the loss;
- computes the **whole gradient** in one sweep, reusing work instead of repeating it (unlike finite differences);
- is built from matrix operations, so it runs fast on GPUs.

That algorithm is **backpropagation**. We'll build it from one example: follow one hidden weight's path to the loss, and see what the chain rule needs along the way.


### Step 2: a hidden weight, by the chain rule

**This step's entry of the gradient** ([listed at the top of this section](#section-5)):

$$
\dfrac{\partial L}{\partial w^{(1)}_{21}}
$$

Backprop starts with a **forward pass**, and **caches** (stores) the values it computes along the way: the hidden outputs $h_1, h_2$ and the output $p$. You'll see in a moment why it needs them.

Take one hidden weight: $w^{(1)}_{21}$, the weight from input $x_1$ into hidden unit 2. It affects the loss along one path through the network, highlighted below. It's the **only** path: $w^{(1)}_{21}$ changes only $h_2$, and $h_2$ feeds only $p$. The output also depends on $h_1$, but $h_1$ doesn't depend on $w^{(1)}_{21}$, so it adds nothing to this slope. The path has five links, one per operation: each edge (a weighted sum), each sigmoid, and the loss. They're numbered from the loss backward, the order the chain rule multiplies them:

- **Link 1** (the arrow out of $p$, the loss): the output $p$ changes the loss $L$.
- **Link 2** (inside the output unit, its sigmoid): the output score $z^{(2)}$ changes the output $p$.
- **Link 3** (the edge $h_2 \to p$, weight $w^{(2)}_2$): hidden unit 2's output $h_2$ changes the output score $z^{(2)}$.
- **Link 4** (inside hidden unit 2, its sigmoid): hidden unit 2's score $z^{(1)}_2$ changes its output $h_2$.
- **Link 5** (the edge $x_1 \to h_2$, weight $w^{(1)}_{21}$): the weight $w^{(1)}_{21}$ changes hidden unit 2's score $z^{(1)}_2$.


In [ ]:
from ch_6_and_10_demos_helpers import plot_backprop_path

plot_backprop_path(general_loss=True);

Written as functions inside functions, one function per link (holding $h_1$ and the other weights fixed):

$$
L\bigg(p\Big(z^{(2)}\Big(h_2\big(z^{(1)}_2(w^{(1)}_{21})\big)\Big)\Big)\bigg)
$$

The chain rule peels them off from the outside in, one local derivative per link:

$$
\frac{\partial L}{\partial w^{(1)}_{21}} = \frac{\partial L}{\partial p} \cdot \frac{\partial p}{\partial z^{(2)}} \cdot \frac{\partial z^{(2)}}{\partial h_2} \cdot \frac{\partial h_2}{\partial z^{(1)}_2} \cdot \frac{\partial z^{(1)}_2}{\partial w^{(1)}_{21}}
$$

Each local derivative follows the rule for its kind of operation, the same three kinds as in Step 1:

$$
\begin{aligned}
\frac{\partial L}{\partial p} & && \text{the loss: left general, as in Step 1} \\
\frac{\partial p}{\partial z^{(2)}} &= p\,(1 - p) && \text{a sigmoid: output} \times (1 - \text{output}) \\
\frac{\partial z^{(2)}}{\partial h_2} &= w^{(2)}_2 && \text{a weighted sum, by an input: the weight that multiplies it} \\
\frac{\partial h_2}{\partial z^{(1)}_2} &= h_2\,(1 - h_2) && \text{a sigmoid: output} \times (1 - \text{output}) \\
\frac{\partial z^{(1)}_2}{\partial w^{(1)}_{21}} &= x_1 && \text{a weighted sum, by a weight: the input that weight multiplies}
\end{aligned}
$$

Links 1 and 2 multiply to $\delta^{(2)}$, the output's error signal from Step 1. Plugging in:

$$
\frac{\partial L}{\partial w^{(1)}_{21}} = \underbrace{\delta^{(2)}}_{\substack{\text{error signal} \\ \text{(links 1, 2)}}}\;\; \underbrace{w^{(2)}_2}_{\substack{\text{edge weight} \\ \text{(link 3)}}}\;\; \underbrace{h_2\,(1 - h_2)}_{\substack{\text{sigmoid derivative} \\ \text{(link 4)}}}\;\; \underbrace{x_1}_{\substack{\text{input} \\ \text{(link 5)}}}
$$

Reading it from left to right, it follows the path backward from the loss: the output's error signal, carried back along the edge from hidden unit 2 to the output (times that edge's weight, $w^{(2)}_2$), through hidden unit 2's sigmoid (times its derivative), to the input $x_1$ that $w^{(1)}_{21}$ multiplies.

**Look at what the formula needs:** $\delta^{(2)}$ (computed from $p$ and the label $y$), $w^{(2)}_2$, $h_2$, and $x_1$. Each is a weight, an input, or built from values the forward pass computed. That's why the forward pass caches them: the backward pass only multiplies numbers that are already in memory.

### Step 3: the whole gradient, and why it's called backpropagation

**This step's entries of the gradient** ([listed at the top of this section](#section-5)), the 5 that Steps 1 and 2 didn't find:

$$
\dfrac{\partial L}{\partial w^{(1)}_{11}},\qquad \dfrac{\partial L}{\partial w^{(1)}_{12}},\qquad \dfrac{\partial L}{\partial w^{(1)}_{22}},\qquad \dfrac{\partial L}{\partial b^{(1)}_1},\qquad \dfrac{\partial L}{\partial b^{(1)}_2}
$$

Each is a substitution into Step 1's or Step 2's result.

**Hidden weights (4 entries, including Step 2's).** The weight from input $x_j$ into hidden unit $k$ has the same kind of path as Step 2's weight: through $h_k$, then $p$, then the loss. So its entry is Step 2's result with $k$ in place of 2 and $j$ in place of 1:

$$
\frac{\partial L}{\partial w^{(1)}_{kj}} = \delta^{(2)}\, w^{(2)}_k\, h_k\,(1 - h_k)\, x_j \qquad \text{for } k = 1, 2 \text{ and } j = 1, 2
$$

**Hidden biases (2 entries).** The same path, except the last local derivative is the bias's instead of the weight's:

$$
\frac{\partial z^{(1)}_k}{\partial b^{(1)}_k} = 1
$$

so $x_j$ becomes 1:

$$
\frac{\partial L}{\partial b^{(1)}_k} = \delta^{(2)}\, w^{(2)}_k\, h_k\,(1 - h_k) \qquad \text{for } k = 1, 2
$$

**Output weights and bias (3 entries),** from Step 1:

$$
\begin{aligned}
\frac{\partial L}{\partial w^{(2)}_k} &= \delta^{(2)}\, h_k \qquad \text{for } k = 1, 2 \\
\frac{\partial L}{\partial b^{(2)}} &= \delta^{(2)}
\end{aligned}
$$

That's $4 + 2 + 2 + 1 = 9$ entries.

**The shared pieces.** Line up the four formulas, from the output back:

$$
\begin{aligned}
\frac{\partial L}{\partial b^{(2)}} &= \delta^{(2)} \\
\frac{\partial L}{\partial w^{(2)}_k} &= \delta^{(2)}\, h_k \\
\frac{\partial L}{\partial b^{(1)}_k} &= \underbrace{\delta^{(2)}\, w^{(2)}_k\, h_k\,(1 - h_k)}_{\delta^{(1)}_k} \\
\frac{\partial L}{\partial w^{(1)}_{kj}} &= \underbrace{\delta^{(2)}\, w^{(2)}_k\, h_k\,(1 - h_k)}_{\delta^{(1)}_k}\, x_j
\end{aligned}
$$

- $\delta^{(2)}$ appears in all 9 entries.
- $\delta^{(2)}\, w^{(2)}_k\, h_k(1 - h_k)$ appears in all three of hidden unit $k$'s entries. Call it $\delta^{(1)}_k$, hidden unit $k$'s **error signal**. Like $\delta^{(2)}$, it's the slope of the loss with respect to its unit's score: $\delta^{(1)}_k = \partial L / \partial z^{(1)}_k$.

So every entry of the gradient is **(the error signal of the unit the parameter feeds into) × (the input the parameter multiplies)**: for an output weight that input is $h_k$, for a hidden weight it's $x_j$, and for a bias it's 1.

**Why "back" propagation.** Compute $\delta^{(2)}$ first, at the output. Then compute each $\delta^{(1)}_k$ from it: $\delta^{(2)}$, times the weight on the edge back to hidden unit $k$, times that unit's sigmoid derivative. The error signals are computed from the output back toward the input, the reverse of the forward pass. With more layers, the same step repeats, one layer further back each time.

**Why it's fast.** The backward pass reuses $h_1$, $h_2$, and $p$ from the forward pass, and each error signal is computed once and shared by every entry that contains it. The backward pass does the same kinds of matrix multiplications as the forward pass, about one and a half times as many, so the whole gradient costs only a few forward passes' worth of arithmetic, however many parameters the network has. Compare finite differences: two forward passes **per parameter**.

**Any network.** Nothing here depended on the formulas from Steps 1 and 2. Each piece of the network supplies only its own local derivative (a weighted sum's is its input or its weight; an activation's is its derivative), and backprop multiplies them along the paths from the loss backward. Libraries run this recipe automatically for whatever network you build: PyTorch's `loss.backward()`, and the training inside scikit-learn's `MLPClassifier.fit()`, which you'll use Friday.

**Back to all the data.** For the full loss, each entry of the gradient is the average of these one-point entries over the $n$ data points.

### Step 4: plugging in the loss

Everything in Steps 1 to 3 holds for any loss. The loss enters only through $\delta^{(2)}$. Our network uses Section 2's log-loss:

$$
L = -\big[\, y \ln p + (1 - y)\ln(1 - p) \big]
$$

Its local derivative, from the derivative of $\ln$ applied to each term:

$$
\frac{\partial L}{\partial p} = -\frac{y}{p} + \frac{1 - y}{1 - p}
$$

Plugging this into $\delta^{(2)} = \dfrac{\partial L}{\partial p}\, p\,(1 - p)$ from Step 1:

$$
\begin{aligned}
\delta^{(2)} &= \left(-\frac{y}{p} + \frac{1 - y}{1 - p}\right) p\,(1 - p) \\
&= -y\,(1 - p) + (1 - y)\,p \\
&= \underbrace{p - y}_{\text{prediction error}}
\end{aligned}
$$

The sigmoid's derivative cancels the log's denominators, leaving the **prediction error**: one subtraction, since $p$ comes from the forward pass and $y$ is the label. With it, Step 1's slopes are

$$
\begin{aligned}
\frac{\partial L}{\partial w^{(2)}_k} &= (p - y)\, h_k \\
\frac{\partial L}{\partial b^{(2)}} &= p - y
\end{aligned}
$$

Compare logistic regression's slopes from [Chapter 4, Section 10](../ch_4/gradient_descent_demos.ipynb#section-10). There, the model had one feature $x$ and the score $z = \beta_0 + \beta_1 x$, and the slopes were

$$
\begin{aligned}
\frac{\partial L}{\partial \beta_1} &= \frac{1}{n}\sum_{i=1}^{n} \big(p_i - y_i\big)\,x_i \\
\frac{\partial L}{\partial \beta_0} &= \frac{1}{n}\sum_{i=1}^{n} \big(p_i - y_i\big)
\end{aligned}
$$

Each is an average of **(prediction error) × (feature)**, where the intercept's "feature" is the constant 1. For one data point, they're $(p - y)\,x$ and $p - y$: exactly Step 1's slopes, with $h_k$ in place of $x$ and $b^{(2)}$ in place of $\beta_0$. **The output unit is a logistic regression whose features are the hidden outputs.** Chapter 4 stated its slopes without deriving them; this is the derivation. In Lab 3 you wrote them as `grad = X.T @ (p - y) / n`, and for all the data at once, the output layer's code is that line with `H` in place of `X`.

**A different loss changes only $\delta^{(2)}$.** Steps 1 to 3 stay the same. Monday looks at the losses used for other kinds of output.


**Code worth reading.** Backpropagation for every data point at once. Each line is one line of the math, using Monday's shapes: one row per data point.

| Math (one data point) | Code (all data points) | Shape |
|---|---|---|
| $\delta^{(2)} = p - y$ (Step 4) | `D2 = (p - y)[:, None]` | (n, 1): one row per data point, one column per output unit (we have one) |
| $\partial L / \partial w^{(2)}_k = \delta^{(2)} h_k$, averaged | `D2.T @ H / n` | (1, 2), the same shape as `W2` |
| $\delta^{(1)}_k = \delta^{(2)} w^{(2)}_k\, h_k(1 - h_k)$ | `D1 = (D2 @ W2) * H * (1 - H)` | (n, 2): one row per data point, one column per hidden unit |
| $\partial L / \partial w^{(1)}_{kj} = \delta^{(1)}_k x_j$, averaged | `D1.T @ X / n` | (2, 2), the same shape as `W1` |

`D2.T @ H / n` is Lab 3's `X.T @ (p - y) / n` with `H` in place of `X`. In `D2 @ W2`, each data point's output error is multiplied by each output weight: row $i$ is $\big(\delta^{(2)} w^{(2)}_1,\ \delta^{(2)} w^{(2)}_2\big)$ for point $i$, the error carried back along each edge. The `*` is entry by entry. **Only the first line depends on the loss:** with a different loss, `D2` changes and the other three lines stay the same.

In [ ]:
def backprop(params, X, y):
    "The slope of the loss for every parameter, by backpropagation."
    n = len(y)
    H, p = forward(params, X)                          # the forward pass
    D2 = (p - y)[:, None]                              # output error signal, shape (n, 1)
    D1 = (D2 @ params['W2']) * H * (1 - H)             # hidden error signals, shape (n, 2)
    return {'W1': D1.T @ X / n,                        # each slope: error signal x input, averaged
            'b1': D1.mean(axis=0),
            'W2': D2.T @ H / n,
            'b2': D2.mean(axis=0)}


exact = backprop(params, X_xor, y_xor)
for name in params:
    print(name, exact[name].round(4))

**The check.** Backpropagation and finite differences should agree. Here is the largest difference between them, over all 9 entries of the gradient:

In [ ]:
largest = max(np.abs(exact[name] - numerical[name]).max() for name in params)
print(f'largest difference: {largest:.1e}')

They agree to about ten decimal places, the size of finite differences' own rounding error. The derivation, and its code, are right.

<a id="section-6"></a>

## 6 · Training

Now the gradient descent loop is the one from [Chapter 4, Section 10](../ch_4/gradient_descent_demos.ipynb#section-10), with backpropagation supplying the slopes. Each step moves every parameter against its slope.

**Code worth reading.** The training loop. It records the loss at every step, and keeps a copy of the parameters at a few chosen steps, for the pictures.

In [ ]:
def train(params, X, y, eta=1.0, steps=3000, keep=()):
    "Gradient descent. Returns the trained parameters, the loss at each step, and snapshots."
    params = {name: value.copy() for name, value in params.items()}   # don't mutate the input
    losses, snapshots = [], {}
    for t in range(steps):
        if t in keep:
            snapshots[t] = {name: value.copy() for name, value in params.items()}
        losses.append(loss(params, X, y))
        grads = backprop(params, X, y)
        for name in params:
            params[name] = params[name] - eta * grads[name]       # the update rule
    snapshots[steps] = params
    return params, losses, snapshots


def accuracy(params, X, y):
    _, p = forward(params, X)
    return np.mean((p > 0.5) == (y == 1))


trained, losses, snapshots = train(random_params(seed=1), X_xor, y_xor, keep=(0, 1000, 2200, 2600))
print('loss: start', round(losses[0], 3), ' step 1000', round(losses[1000], 3),
      ' step 2200', round(losses[2200], 3), ' end', round(losses[-1], 3))
print('accuracy at the end:', accuracy(trained, X_xor, y_xor))

In [ ]:
from ch_6_and_10_demos_helpers import plot_training_run

plot_training_run(losses, snapshots, X_xor, y_xor);

**Reading the run.** Each lower panel shades the network's probability $p$: blue where it's near 1 (class 1), orange where it's near 0 (class 0). The black curve is where $p = 0.5$: the decision boundary. Circled points are misclassified.

- **First, a quick partial fix.** By step 1,000 the loss has dropped from 0.701 to about 0.50, and the boundary has started to bend, but it still misclassifies a whole cluster.
- **Then a long plateau.** From step 1,000 to about step 2,300, almost nothing happens: the loss creeps from 0.496 to 0.481. The slopes are tiny here, so the steps are tiny.
- **Then the loss drops fast.** It falls below 0.1 by step 2,519, the boundary bends into the XOR shape, and every point ends up classified correctly.

A plateau is why "the loss stopped changing" isn't a safe rule for when to stop training a network.

### Different starting weights, different outcomes

Logistic regression's loss is a single bowl, so gradient descent reaches the same bottom from any start. A network's loss isn't a bowl. Here are 10 runs, each from different random starting weights, with everything else the same:

In [ ]:
from ch_6_and_10_demos_helpers import plot_many_runs

curves, accuracies = [], []
for seed in range(10):
    trained_s, losses_s, _ = train(random_params(seed), X_xor, y_xor)
    curves.append(losses_s)
    accuracies.append(accuracy(trained_s, X_xor, y_xor))

plot_many_runs(curves, accuracies);

Most runs solve XOR. The ones that don't get **stuck**: their loss levels off around 0.35, with only 53% to 62% of the points classified correctly. Gradient descent only sees the ground under its feet, so it can settle in a dip that isn't the lowest point. The usual remedies are to train again from new random weights, or to give the network more hidden units, which gives gradient descent more ways downhill.

**Ask:** Your network's loss has been flat for 1,000 steps. Is it stuck, or on a plateau? How could you tell?


<details>
<summary>Answer</summary>

From the loss curve alone, you can't tell yet: seed 1's plateau looked exactly like a stuck run for over a thousand steps (from step 1,000 to about step 2,300).

Ways to find out:

- **Train longer.** A plateau ends; a stuck run stays flat.
- **Check the accuracy and the boundary.** A stuck run here classifies only about half the points correctly, which you can see in the picture.
- **Train again from other random starts.** If other runs reach a much lower loss, this one is stuck.

</details>

<a id="section-7"></a>

## 7 · Minibatches

Every step above used all 100 data points to compute the slopes. Real training sets have millions of points, so one full-data step is expensive. **Minibatch gradient descent** computes each step's slopes from a small random batch of points instead:

1. Shuffle the data.
2. Split it into batches, of 10 points, say (in this example, in real life you don't need batches this small).
3. Take one gradient descent step per batch, using only that batch's points.

One pass through all the data is an **epoch**. With 100 points and batches of 10, an epoch is 10 steps instead of 1, for the same amount of computation. Each step is noisier, because 10 points only estimate the slopes of the whole loss. But there are many more steps.

**Why count passes?** Training a network takes many passes through the data, not one. You've already done this: in Lab 3 and in Section 6, every gradient descent step used all the data, so each step *was* one pass, and Section 6's 3,000 steps were 3,000 passes. Gradient descent needs many passes because each step moves the parameters only a little (the learning rate $\eta$ sets how far), and there's no formula that jumps straight to the best parameters, the way there is for linear regression. With full-data steps, counting steps and counting passes give the same number. With minibatches they don't, so passes get their own name: epochs. (Many models you've fit with scikit-learn also iterate like this behind the scenes. `LogisticRegression`'s `max_iter` is the same idea.)

When the batch size is 1, it's called **stochastic gradient descent** (SGD), and many people (and libraries) say "SGD" for any minibatch size.

**Code worth reading.** The minibatch loop. Only the inner loop over batches is new. `backprop` is unchanged: it just receives fewer rows. The comparison below starts from the same weights as the plateau run in Section 6 (seed 1).

In [ ]:
def train_minibatch(params, X, y, eta=1.0, epochs=300, batch_size=10, seed=0):
    "Minibatch gradient descent. Returns the trained parameters and the loss after each epoch."
    params = {name: value.copy() for name, value in params.items()}
    rng = np.random.default_rng(seed)
    losses = []
    for epoch in range(epochs):
        order = rng.permutation(len(y))                    # 1. shuffle
        for start in range(0, len(y), batch_size):         # 2. one batch at a time
            batch = order[start:start + batch_size]
            grads = backprop(params, X[batch], y[batch])   # slopes from this batch only
            for name in params:
                params[name] = params[name] - eta * grads[name]   # 3. one step per batch
        losses.append(loss(params, X, y))                  # the loss on all the data
    return params, losses


start = random_params(seed=1)
_, full_losses, _ = train(start, X_xor, y_xor, steps=300)
mini_trained, mini_losses = train_minibatch(start, X_xor, y_xor, epochs=300, batch_size=10)
print('after 300 epochs: full batch', round(full_losses[-1], 3), '  minibatch', round(mini_losses[-1], 3))

In [ ]:
from ch_6_and_10_demos_helpers import plot_full_vs_minibatch

plot_full_vs_minibatch(full_losses, mini_losses, batch_size=10, n=n);

For the same 300 epochs of computation, the minibatch run took 3,000 small, noisy steps and solved the problem: its loss is 0.004. The full-batch run took 300 careful steps, and its loss is still 0.676, near where it started. With large data sets this trade-off is even more lopsided, which is why essentially every network is trained with minibatches.

<a id="summary"></a>

## 8 · Summary

- **The loss is logistic regression's.** A network with one sigmoid output is scored with the log-loss, averaged over the data points.
- **Gradient descent is the loop; backpropagation computes the slopes.** The update rule is Chapter 4's: new value = old value − η × slope.
- **Finite differences** measures a slope by nudging a weight both ways. It's correct but needs two forward passes per parameter, so it's used to *check* slope formulas, not to train.
- **Backpropagation** is the chain rule. The loss enters only through the output's error signal $\delta^{(2)}$; for log-loss it's the prediction error $p - y$, and the output layer's slopes are logistic regression's, with the hidden outputs as the features. Each hidden weight's slope multiplies the local derivatives along its path back from the loss. Every slope is (error signal at the unit) × (input), and the error signals are computed from the output backward.
- **A network's loss isn't a bowl.** Training can sit on a plateau for a long time and then drop, or get stuck, depending on the starting weights.
- **Minibatches** trade exact, expensive steps for many cheap, noisy ones. An epoch is one pass through the data.

**On Friday** you'll train networks with scikit-learn's `MLPClassifier`. Its settings are today's ideas: `hidden_layer_sizes` (how many hidden units), `activation`, `batch_size`, `learning_rate_init` (η), `max_iter` (epochs), and, after fitting, `loss_curve_` (the loss after each epoch).